# Hallucination Observation & Mitigation

## Hallucination in LLMs: Observation & Mitigation

### What is Hallucination?

**Hallucination** in a Large Language Model (LLM) occurs when the model generates information that sounds plausible but is **factually incorrect, fabricated, misleading, or unsupported by the input or reliable knowledge**.

**Example:**

* **Question:** Who invented the telephone?
* **Correct Answer:** Alexander Graham Bell.
* **Hallucinated Answer:** Thomas Edison invented the telephone in 1880.

The hallucinated answer is fluent but factually wrong.

---

## Why does hallucination happen in LLMs?

Hallucination occurs because LLMs **predict the most likely next word** based on patterns learned during training, rather than verifying facts. It can happen due to:

* **Incomplete or outdated training data**
* **Lack of real-time fact-checking**
* **Ambiguous or unclear prompts**
* **Insufficient context**
* **Overgeneralization from learned patterns**

As a result, the model may generate **plausible-sounding but incorrect or fabricated information**.

## 1. Hallucination Observation

Hallucination observation is the process of **detecting, measuring, and analyzing** when and why an LLM produces incorrect or fabricated information.

### Methods of Observation

* **Fact Verification:** Compare the model's output with trusted sources.
* **Human Evaluation:** Domain experts review responses for accuracy and completeness.
* **Benchmark Testing:** Evaluate models on datasets designed to reveal hallucinations (e.g., TruthfulQA, HaluEval).
* **Confidence Analysis:** Monitor uncertainty or low-confidence responses.
* **Grounding Checks:** Verify whether the answer is supported by the provided context or retrieved documents.
* **Consistency Testing:** Ask the same question in different ways; inconsistent answers may indicate hallucination.

### Indicators of Hallucination

* Invented facts, names, dates, or references.
* Fake citations or URLs.
* Contradictory statements.
* Overconfident answers despite insufficient information.
* Information not supported by the provided context.

---


## 2. Hallucination Mitigation

Hallucination mitigation refers to **techniques used to reduce or prevent incorrect or fabricated outputs** from LLMs.

### Common Mitigation Techniques

### A. Retrieval-Augmented Generation (RAG)

* Retrieve relevant information from trusted databases or documents before generating an answer.
* The model bases its response on retrieved evidence instead of relying only on internal knowledge.

**Benefit:** Reduces factual errors and improves accuracy.

---

### B. Prompt Engineering

Design prompts that encourage truthful behavior.

**Examples:**

* "Answer only using the provided context."
* "If you don't know the answer, say 'I don't know.'"
* "Provide evidence for each claim."

---

### C. Fine-Tuning

Train the model on:

* High-quality datasets
* Domain-specific knowledge
* Human feedback (RLHF)

This helps the model generate more accurate and reliable responses.

---

### D. Grounding

Require the model to base answers only on:

* Retrieved documents
* Databases
* Knowledge graphs
* External APIs

Grounded responses are less likely to contain fabricated information.

---

### E. Human-in-the-Loop (HITL)

Have humans review AI-generated responses before they are used in critical applications such as:

* Healthcare
* Legal services
* Finance
* Scientific research

---

### F. Output Validation

Automatically check generated responses by:

* Fact-checking against trusted sources
* Detecting contradictions
* Validating numerical calculations
* Verifying references and citations

---

### G. Confidence Thresholding

If the model has low confidence:

* Ask for clarification.
* Respond with uncertainty.
* Escalate to a human reviewer when appropriate.

---

### H. Chain-of-Verification (CoVe)

After generating an answer, the model performs a verification step by checking important claims against available evidence before presenting the final response.

---

## Observation vs. Mitigation

| Aspect     | Hallucination Observation                                  | Hallucination Mitigation                                                 |
| ---------- | ---------------------------------------------------------- | ------------------------------------------------------------------------ |
| Purpose    | Detect hallucinations                                      | Prevent or reduce hallucinations                                         |
| Focus      | Evaluation and monitoring                                  | Improving response accuracy                                              |
| Techniques | Fact-checking, benchmarks, human review, consistency tests | RAG, prompt engineering, fine-tuning, grounding, HITL, output validation |
| Outcome    | Identifies when hallucinations occur                       | Produces more reliable and trustworthy outputs                           |

---

## Real-World Example

**User Question:** "What is the capital of Australia?"

* **Hallucinated Response:** Sydney.
* **Observed Issue:** Fact-checking identifies the answer as incorrect.
* **Mitigation:** A RAG system retrieves verified information from a trusted source, allowing the model to answer correctly: **Canberra**.

---

## Key Takeaways

* **Hallucination** is the generation of false, fabricated, or unsupported information by an LLM.
* **Observation** focuses on detecting and measuring hallucinations using evaluation techniques such as fact-checking, benchmarking, and human review.
* **Mitigation** focuses on reducing hallucinations through methods like Retrieval-Augmented Generation (RAG), prompt engineering, grounding, fine-tuning, output validation, and human oversight.
* Combining observation and mitigation leads to more accurate, reliable, and trustworthy LLM applications.


##### Good pairing — one shows how hallucinations get induced/observed, the other shows a concrete fix.

## 1. Trigger hallucination with an adversarial prompt

The trick isn't the API call — it's the prompt design: false premise ("System 3" doesn't exist), authority framing, or asking for an oddly specific detail (exact page/quote) that pressures the model to fabricate rather than say "I don't know."

In [10]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

def ask(prompt, temperature=1.0):
    resp = client.chat.completions.create(
        model="gpt-4o",
        temperature=temperature,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

# Adversarial prompt: embeds a false premise + citation bait
prompt = "What page of Kahneman's 'Thinking, Fast and Slow' defines 'System 3' thinking?"

# Sample the same prompt multiple times — a model that "knows" the fact
# repeats it; a model that's confabulating gives a different fake
# page number / definition each time.
for i in range(3):
    print(f"Sample {i+1}: {ask(prompt)}\n")

Sample 1: In Daniel Kahneman's book "Thinking, Fast and Slow," there isn't a concept referred to as "System 3" thinking. The book primarily discusses two systems of thought: System 1, which is fast, intuitive, and automatic; and System 2, which is slower, more deliberate, and analytical. If you are looking for information about a third system of thought, it might be from a different work or author, or possibly a misinterpretation or extension of Kahneman's concepts.

Sample 2: In Daniel Kahneman's book "Thinking, Fast and Slow," there is no mention of a 'System 3' thinking. The book primarily discusses two systems: System 1, which is fast, automatic, and intuitive; and System 2, which is slower, more deliberate, and analytical. The idea of a 'System 3' is not covered in this work. If you have further questions about the concepts covered in the book, feel free to ask!

Sample 3: In Daniel Kahneman's "Thinking, Fast and Slow," there is no mention of a 'System 3' thinking. The book primar

## 2. Grounding with source attribution

### Core idea, in one line each:

- **Trigger:** craft a prompt with a false premise or an oddly specific ask, then resample — inconsistency across samples = hallucination signal.
- **Ground:** retrieve source text, compute similarity between each generated sentence and the sources, flag anything below threshold as unsupported.

In [12]:
from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = OpenAI()

sources = [
    "Kahneman describes System 1 (fast, intuitive) and System 2 (slow, deliberate). The book does not define a System 3.",
    "The Eiffel Tower is 330 meters tall, completed in 1889.",
]

def draft_answer(query, context):
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": f"Using this context, answer in 1-2 sentences:\n{context}\n\nQuestion: {query}"
        }],
    )
    return resp.choices[0].message.content

def attribute(sentence, sources, threshold=0.15):
    vec = TfidfVectorizer(stop_words="english").fit(sources + [sentence])
    sims = cosine_similarity(vec.transform([sentence]), vec.transform(sources))[0]
    best = sims.argmax()
    if sims[best] >= threshold:
        return f"✅ Supported by source {best} (sim={sims[best]:.2f})"
    return f"⚠️ UNSUPPORTED (sim={sims[best]:.2f})"

# Generate grounded, then verify claim-by-claim
answer = draft_answer("Does Kahneman's book define a System 3?", "\n".join(sources))
claims = answer.split(". ")  # crude sentence split for demo

for c in claims:
    if c.strip():
        print(c.strip(), "->", attribute(c, sources))

No, Kahneman's book does not define a System 3; it only describes System 1 and System 2. -> ✅ Supported by source 0 (sim=0.65)
